# Pool real HAM10000 MEL images into training (Correction 2)


Needs GPU. Uses `expA_train/val/test.csv` + ISIC images (unchanged) plus `HAM10000_metadata.csv` and the HAM10000 image folder. Splits HAM10000 by lesion_id into a pool-in-training set and a disjoint external-holdout set. Edit the five `EDIT THESE` paths below.

In [1]:
# ============================================================
# Pool real HAM10000 MEL images into the ISIC 2019 training set,
# to answer the correction that "a large amount of [real] data
# is required" -- as a real-data alternative to the GAN-synthetic
# MEL pipeline, which was already shown to fail (mode collapse).
#
# DATASETS NEEDED (edit the paths below):
#   - Your existing ISIC expA_train.csv / expA_val.csv /
#     expA_test.csv and LABELED_DIR (ISIC images), unchanged.
#   - Pretrained SSL encoder weights (SSL_ENCODER_CKPT), same
#     Stage-1 checkpoint used in your other fine-tuning notebooks.
#   - HAM10000_metadata.csv (has columns: lesion_id, image_id,
#     dx, ...). Point HAM_META_CSV at it.
#   - The HAM10000 image folder(s) (HAM10000_images_part_1 and
#     _part_2, or wherever you have them merged/mounted). Point
#     HAM_IMAGE_DIR at the parent folder (searched recursively).
#
# WHAT THIS DOES:
#   1. Filters HAM10000 to the three shared classes (bkl/mel/nv
#      -> BKL/MEL/NV) and groups by lesion_id (a lesion can have
#      several images -- the split is done by LESION, never by
#      image, so no lesion appears in both the pooled-training
#      set and the held-out external test set -- this avoids the
#      leakage you'd get from splitting by image_id alone).
#   2. Randomly assigns HAM10000 MEL lesions into "pool-in"
#      (added to ISIC training data) vs. "external-holdout" (kept
#      as a clean, disjoint external test set), using
#      HAM_POOL_FRACTION of MEL lesions. By default only the
#      scarce MEL class is pooled (set ALSO_POOL_MAJORITY_CLASSES
#      = True to also pool BKL/NV lesions the same way).
#   3. Fine-tunes the SAME pipeline (5x MEL oversampling, heavy
#      MEL augmentation, focal loss 4/2/1, gamma=1.5) on
#      ISIC-train + pooled-in HAM10000 MEL images, for 3 seeds.
#   4. Evaluates each seed's model on TWO test sets:
#        (a) the original, unchanged 104-image ISIC test set,
#        (b) the disjoint HAM10000 external-holdout set (the
#            lesions that were NOT pooled into training).
#      This gives a clean before/after comparison against your
#      existing "6.6% MEL recall on full HAM10000" number, using
#      a held-out subset the model has genuinely never seen.
#
# OUTPUT:
#   - ham_pool_manifest.csv: which HAM10000 lesions/images went
#     into training vs. stayed in the external holdout (keep this
#     for a reproducibility appendix).
#   - ham_pooling_results.csv: per-seed accuracy/macro-F1/MEL
#     recall/MEL precision on both the ISIC test set and the new
#     HAM10000 external-holdout set. The key comparison for the
#     paper is ham_mel_recall here vs. the existing 6.6% number.
# ============================================================
import os, glob, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import f1_score
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- EDIT THESE ----
BACKUP_DIR       = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
LABELED_DIR      = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
SSL_ENCODER_CKPT = '/kaggle/input/datasets/ittisamurtunib/ssl-encoder-real-24000/ssl_encoder_real_24000.pth'
HAM_META_CSV     = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'

import glob

# Collect all image paths across both subdirectories
ham_image_paths = glob.glob('/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_*/*.jpg')

# Build mapping: 'ISIC_0024306' -> '/kaggle/input/.../HAM10000_images_part_1/ISIC_0024306.jpg'
HAM_IMAGE_MAP = {os.path.splitext(os.path.basename(p))[0]: p for p in ham_image_paths}

# ---------------------

OUTPUT_DIR = '/kaggle/working'
SEEDS = [42, 123, 2024]
CLASSES = ['BKL', 'MEL', 'NV']
DX_MAP = {'bkl': 'BKL', 'mel': 'MEL', 'nv': 'NV'}
HAM_POOL_FRACTION = 0.5             # fraction of MEL lesions pooled into training
ALSO_POOL_MAJORITY_CLASSES = False  # set True to also pool BKL/NV lesions
MEL_OVERSAMPLE = 5
MAX_EPOCHS = 30
PATIENCE = 7

# ---------- 1. Build HAM10000 pool / holdout split (lesion-level) ----------
ham_meta = pd.read_csv(HAM_META_CSV)
ham_meta = ham_meta[ham_meta['dx'].isin(DX_MAP.keys())].copy()
ham_meta['label'] = ham_meta['dx'].map(DX_MAP)

rng = np.random.default_rng(42)


def split_lesions(df, fraction):
    lesions = df['lesion_id'].unique().copy()
    rng.shuffle(lesions)
    n_pool = int(len(lesions) * fraction)
    pool_lesions = set(lesions[:n_pool])
    return df[df['lesion_id'].isin(pool_lesions)], df[~df['lesion_id'].isin(pool_lesions)]


mel_df = ham_meta[ham_meta['label'] == 'MEL']
mel_pool, mel_holdout = split_lesions(mel_df, HAM_POOL_FRACTION)

if ALSO_POOL_MAJORITY_CLASSES:
    other_df = ham_meta[ham_meta['label'] != 'MEL']
    other_pool, other_holdout = split_lesions(other_df, HAM_POOL_FRACTION)
    pool_df_ham = pd.concat([mel_pool, other_pool], ignore_index=True)
    holdout_df = pd.concat([mel_holdout, other_holdout], ignore_index=True)
else:
    pool_df_ham = mel_pool
    holdout_df = ham_meta[~ham_meta['lesion_id'].isin(set(mel_pool['lesion_id']))]

manifest = pd.concat([
    pool_df_ham.assign(split='pool_in_training'),
    holdout_df.assign(split='external_holdout'),
], ignore_index=True)[['lesion_id', 'image_id', 'label', 'split']]
manifest.to_csv(f'{OUTPUT_DIR}/ham_pool_manifest.csv', index=False)
print(f"HAM10000 MEL lesions pooled into training: {mel_pool['lesion_id'].nunique()} "
      f"({len(mel_pool)} images)")
print(f"HAM10000 MEL lesions in external holdout:  {mel_holdout['lesion_id'].nunique()} "
      f"({len(mel_holdout)} images)")

# ---------- 2. Assemble augmented training set ----------
isic_train = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_train.csv'))
isic_val = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
isic_test = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))
isic_train['source'] = 'isic'
isic_val['source'] = 'isic'
isic_test['source'] = 'isic'

ham_pool_rows = pool_df_ham.rename(columns={'image_id': 'image'})[['image', 'label']].copy()
ham_pool_rows['source'] = 'ham'
train_augmented = pd.concat([isic_train, ham_pool_rows], ignore_index=True)

ham_holdout_rows = holdout_df.rename(columns={'image_id': 'image'})[['image', 'label']].copy()
ham_holdout_rows['source'] = 'ham'

print(f"\nAugmented training set: {len(train_augmented)} images "
      f"(was {len(isic_train)}, +{len(ham_pool_rows)} real HAM10000 MEL)")

# ---------- 3. Model / data pipeline (same recipe as main results) ----------
train_transform_std = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_transform_mel = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45), transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def resolve_path(row):
    img_name = str(row['image'])
    # Strip extension if present to get clean image_id
    image_id = os.path.splitext(img_name)[0]
    
    if row['source'] == 'isic':
        if not img_name.endswith('.jpg'):
            img_name += '.jpg'
        return os.path.join(LABELED_DIR, img_name)
    else:
        # Fast lookup directly from HAM_IMAGE_MAP
        return HAM_IMAGE_MAP[image_id]


class LesionDataset(Dataset):
    def __init__(self, df, training):
        self.df = df.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Opens image via the O(1) map lookup
        img_path = resolve_path(row)
        img = Image.open(img_path).convert('RGB')
        
        # Apply appropriate transformations
        tfm = (train_transform_mel if row['label'] == 'MEL' else train_transform_std) if self.training else eval_transform
        
        return tfm(img), row['label']


def make_oversampled_df(df):
    mel = df[df['label'] == 'MEL']
    rest = df[df['label'] != 'MEL']
    return pd.concat([rest] + [mel] * MEL_OVERSAMPLE, ignore_index=True)


class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )

    def forward(self, x):
        return self.features(x)


class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, x):
        return self.classifier(self.encoder(x))


def load_encoder_ckpt(encoder, ckpt_path):
    state = torch.load(ckpt_path, map_location=device)
    stripped = {(k[len('encoder.'):] if k.startswith('encoder.') else k): v for k, v in state.items()}
    missing, unexpected = encoder.load_state_dict(stripped, strict=False)
    if missing or unexpected:
        print(f"  [warning] encoder load: {len(missing)} missing, {len(unexpected)} unexpected keys "
              f"-- verify SSL_ENCODER_CKPT points at the right file.")
    return encoder


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=1.5):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, logits, targets):
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()
        logp_t = logp.gather(1, targets.unsqueeze(1)).squeeze(1)
        p_t = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        a_t = self.alpha[targets]
        return (-a_t * (1 - p_t) ** self.gamma * logp_t).mean()


def evaluate(model, loader):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds = model(images).argmax(dim=1).cpu().numpy()
            all_pred.extend([CLASSES[p] for p in preds])
            all_true.extend(labels)
    true_arr, pred_arr = np.array(all_true), np.array(all_pred)
    acc = np.mean(pred_arr == true_arr)
    macro_f1 = f1_score(true_arr, pred_arr, labels=CLASSES, average='macro', zero_division=0)
    tp = np.sum((true_arr == 'MEL') & (pred_arr == 'MEL'))
    fn = np.sum((true_arr == 'MEL') & (pred_arr != 'MEL'))
    fp = np.sum((true_arr != 'MEL') & (pred_arr == 'MEL'))
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    return acc, macro_f1, recall, precision


class_to_idx = {c: i for i, c in enumerate(CLASSES)}
results = []

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_loader = DataLoader(LesionDataset(make_oversampled_df(train_augmented), training=True), batch_size=16, shuffle=True)
    val_loader = DataLoader(LesionDataset(isic_val, training=False), batch_size=16, shuffle=False)
    isic_test_loader = DataLoader(LesionDataset(isic_test, training=False), batch_size=16, shuffle=False)
    ham_holdout_loader = DataLoader(LesionDataset(ham_holdout_rows, training=False), batch_size=16, shuffle=False)

    encoder = SimpleEncoder()
    encoder = load_encoder_ckpt(encoder, SSL_ENCODER_CKPT)
    model = SSLClassifier(encoder, len(CLASSES)).to(device)

    alpha = torch.tensor([2.0, 4.0, 1.0], device=device)
    criterion = FocalLoss(alpha, gamma=1.5)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_f1, best_state, no_improve = -1, None, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for images, labels in train_loader:
            images = images.to(device)
            targets = torch.tensor([class_to_idx[l] for l in labels], device=device)
            optimizer.zero_grad()
            loss = criterion(model(images), targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        _, val_f1, _, _ = evaluate(model, val_loader)
        if val_f1 > best_f1:
            best_f1, best_state, no_improve = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    isic_metrics = evaluate(model, isic_test_loader)
    ham_metrics = evaluate(model, ham_holdout_loader)
    print(f"[seed {seed}] ISIC test: acc={isic_metrics[0]:.3f} macroF1={isic_metrics[1]:.3f} "
          f"MELrec={isic_metrics[2]:.3f} | HAM holdout: acc={ham_metrics[0]:.3f} "
          f"macroF1={ham_metrics[1]:.3f} MELrec={ham_metrics[2]:.3f}")
    results.append({'seed': seed,
                     'isic_accuracy': isic_metrics[0], 'isic_macro_f1': isic_metrics[1],
                     'isic_mel_recall': isic_metrics[2], 'isic_mel_precision': isic_metrics[3],
                     'ham_accuracy': ham_metrics[0], 'ham_macro_f1': ham_metrics[1],
                     'ham_mel_recall': ham_metrics[2], 'ham_mel_precision': ham_metrics[3]})

results_df = pd.DataFrame(results)
results_df.to_csv(f'{OUTPUT_DIR}/ham_pooling_results.csv', index=False)
print(f"\nWrote {OUTPUT_DIR}/ham_pooling_results.csv")
print("\nMean +/- SD across 3 seeds (compare ham_mel_recall to the existing 6.6% number):")
for col in ['isic_mel_recall', 'ham_mel_recall', 'isic_macro_f1', 'ham_macro_f1']:
    print(f"  {col:20s}: {results_df[col].mean():.3f} +/- {results_df[col].std():.3f}")


HAM10000 MEL lesions pooled into training: 307 (555 images)
HAM10000 MEL lesions in external holdout:  307 (558 images)

Augmented training set: 1038 images (was 483, +555 real HAM10000 MEL)
[seed 42] ISIC test: acc=0.462 macroF1=0.302 MELrec=1.000 | HAM holdout: acc=0.519 macroF1=0.311 MELrec=0.962
[seed 123] ISIC test: acc=0.596 macroF1=0.487 MELrec=0.750 | HAM holdout: acc=0.574 macroF1=0.432 MELrec=0.812
[seed 2024] ISIC test: acc=0.471 macroF1=0.309 MELrec=1.000 | HAM holdout: acc=0.493 macroF1=0.299 MELrec=0.980

Wrote /kaggle/working/ham_pooling_results.csv

Mean +/- SD across 3 seeds (compare ham_mel_recall to the existing 6.6% number):
  isic_mel_recall     : 0.917 +/- 0.144
  ham_mel_recall      : 0.918 +/- 0.093
  isic_macro_f1       : 0.366 +/- 0.105
  ham_macro_f1        : 0.347 +/- 0.074
